In [10]:
import warnings
from pathlib import Path

import prism

from imagematerials.concepts import knowledge_graph
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.util import (
    export_to_netcdf, import_from_netcdf, rebroadcast_prep_data,
    read_climate_policy_config, read_circular_economy_config
)
from imagematerials.vehicles import (
    preprocess
)
from imagematerials.vehicles.modelling_functions import interpolate
import pandas as pd
from imagematerials.vehicles.constants import years_range


In [4]:
base_dir = "../data/raw"
climate_policy_scenario_dir = Path(base_dir) / 'SSP2'
circular_economy_scenario_dirs = {"narrow": Path(base_dir) / 'circular_economy_scenarios' / 'narrow'}
prep_fp = Path("prep_vema2.nc")

In [5]:
if not prep_fp.is_file():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
        circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
        orig_prep_data = preprocess(base_dir, climate_policy_config, circular_economy_config)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

Missing mode in mileages: ('cars', 'mileage')
Missing mode in mileages: ('bus', 'mileage')
Missing mode in mileages: ('rail', 'mileage')
Missing mode in mileages: ('bicycle', 'mileage')
Missing mode in mileages: ('light_commercial_vehicles', 'mileage')
Missing mode in mileages: ('ships_inland', 'mileage')


In [ ]:
target_year = circular_economy_config['narrow']['vehicles']['target_year']
base_year = circular_economy_config['narrow']['vehicles']['base_year']
mileage_change = circular_economy_config['narrow']['vehicles']['mileage']

mileages: pd.DataFrame = pd.read_csv("../data/raw/vehicles/SSP2_CP/mileages_km_per_year.csv",
        index_col="t")
kilometrage: pd.DataFrame = pd.read_csv("../data/raw/vehicles/SSP2_CP/kilometrage.csv",
        index_col="t")
kilometrage_bus: pd.DataFrame = pd.read_csv("../data/raw/vehicles/SSP2_CP/kilometrage_bus.csv",
        index_col="t")
kilometrage_midi_bus: pd.DataFrame = pd.read_csv("../data/raw/vehicles/SSP2_CP/kilometrage_midi.csv",
        index_col="t")

mileages = mileages.reindex(years_range).interpolate(limit_direction='both')
kilometrage = kilometrage.reindex(years_range).interpolate(limit_direction='both')
kilometrage_bus = kilometrage_bus.reindex(years_range).interpolate(limit_direction='both')
kilometrage_midi_bus = kilometrage_midi_bus.reindex(years_range).interpolate(limit_direction='both')


In [ ]:
def apply_percentage_change(parameter, change, base_year, target_year):
    parameter = parameter[parameter.index <= base_year].copy()
    parameter.loc[target_year] = parameter.loc[base_year]

    for mode, increase in change.items():
        col = (mode, parameter)
        if col in parameter.columns:
            base_val = parameter.loc[base_year, col]
            parameter.loc[target_year, col] = base_val * (1 + increase / 100)
        else:
            print("Missing mode in {parameter}: {col}")

    return interpolate(pd.DataFrame(parameter))

In [ ]:
apply_percentage_change(mileages, mileage_change, base_year, target_year)

TypeError: unhashable type: 'DataFrame'

In [ ]:
mileages

In [ ]:
for mode, increase in mileage_change.items():
    col = (mode, 'mileage')
    if col in mileages.columns:
        base_val = mileages.loc[base_year, col]
        mileages.loc[target_year, col] = base_val * (1 + increase / 100)
    else:
        print(f"Missing mode in mileages: {col}")

mileages = interpolate(pd.DataFrame(mileages))

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
# main_model_normal = GenericMainModel(
#     complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
#     compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=False, 
#     material=material, knowledge_graph=knowledge_graph)

In [ ]:
new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=prep_data["shares"].coords["Type"].values)
new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=prep_data["shares"].coords["Region"].values)
new_prep_data["knowledge_graph"] = knowledge_graph

In [ ]:
# main_model_normal.simulate(simulation_timeline)

In [ ]:
main_model_factory = ModelFactory(
    new_prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).finish()

In [ ]:
main_model_factory.simulate(simulation_timeline)